# RAG Branching Example

A graph that answers questions about this system using RAG on `vector_db/llm_graph_docs`.

**Graph layout:**

```
branch_llm  ──►  check_type (ConditionalNode)
                    │
          ┌─────────┴──────────┐
       coding               general
          │                    │
  coding_retrieval    general_retrieval
          │                    │
    coding_llm          general_llm
```

- **coding** branch: retrieves relevant docs and answers with a code-focused prompt
- **general** branch: retrieves relevant docs and answers with a conceptual explanation prompt

In [1]:
from dotenv import load_dotenv
from IPython.display import Markdown
from openai import OpenAI

from llm_graph.llm.response_functions import OpenAI_response_fn
from llm_graph.factories.llm import create_llm_node
from llm_graph.factories.rag import create_rag_query_pair
from llm_graph.core.nodes import ConditionalNode
from llm_graph.core.graphrunner import GraphRunner
from llm_graph.core.sessionrunner import SessionRunner

In [2]:
load_dotenv()
client = OpenAI()
response_fn = OpenAI_response_fn(client)

## Prompts

In [3]:
branching_prompt = """
You are making a decision on the type of query given.
This will either be a coding query (asking for code examples, implementation details, usage snippets)
or a general query (asking for conceptual explanations, descriptions, or comparisons).

Respond ONLY with valid JSON with key 'query_type' set to either 'coding' or 'general'.

The query is: {user_query}
"""

coding_prompt = """
You are an expert on the llm-graph-engine system.
A user has asked a coding question about this system.
Use the retrieved documentation to give a clear, complete code example that answers the query.
Include imports where relevant. Do not go beyond what is asked.

Respond ONLY with valid JSON with key 'answer' containing your response.

The query is: {user_query}
"""

general_prompt = """
You are an expert on the llm-graph-engine system.
A user has asked a general question about this system.
Use the retrieved documentation to give a clear conceptual explanation.

Respond ONLY with valid JSON with key 'answer' containing your response.

The query is: {user_query}
"""

## Build the graph

In [4]:
# Classifies the query — routes to 'coding' or 'general'
branch_node = create_llm_node(
    response_fn=response_fn,
    name="branch_llm",
    prompt_template=branching_prompt,
    next_node_name="check_type",
)

check_type_node = ConditionalNode(
    name="check_type",
    condition_fn=lambda state: state["query_type"],
)

In [5]:
DB_PATH = "../vector_db"
COLLECTION = "llm_graph_docs"

# Coding branch: retrieval → LLM with coding prompt
coding_nodes = create_rag_query_pair(
    path=DB_PATH,
    collection_name=COLLECTION,
    response_fn=response_fn,
    retrieval_node_name="coding",
    llm_node_name="coding_llm",
    prompt_template=coding_prompt,
)

# General branch: retrieval → LLM with general prompt
general_nodes = create_rag_query_pair(
    path=DB_PATH,
    collection_name=COLLECTION,
    response_fn=response_fn,
    retrieval_node_name="general",
    llm_node_name="general_llm",
    prompt_template=general_prompt,
)

In [6]:
graphrunner = GraphRunner.build(
    node_dicts=[
        {"branch_llm": branch_node, "check_type": check_type_node},
        coding_nodes,
        general_nodes,
    ],
    start_node="branch_llm",
)

session = SessionRunner(
    graph=graphrunner,
    session_keys=["message_history"],
)

## Run some queries

In [7]:
# General question — should route to the 'general' branch
r1 = session.execute({"user_query": "What is the difference between GraphRunner and SessionRunner?"})
Markdown(r1["state_dict"]["answer"])

c:\Users\ronsp\micromamba\envs\llm_graph\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3250.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GraphRunner executes a graph for a single run and does not retain its `state_dict` between calls to `execute()`.

SessionRunner wraps a GraphRunner to support multiple executions while preserving selected state across runs. Specifically, it:
- Reuses the same GraphRunner instance across calls
- Maintains a persistent `session_dict` containing only the keys listed in `session_keys` (e.g., `message_history`)
- Accumulates `trace_log` outputs from each run in `trace_logs`

In short: use GraphRunner for one-off, stateless executions; use SessionRunner when you need multi-turn/session behavior with state carried over between executions.

In [8]:
# Coding question — should route to the 'coding' branch
r2 = session.execute({"user_query": "How do I create an LLMCall node with a custom prompt template?"})
Markdown(r2["state_dict"]["answer"])

```python
from openai import OpenAI
from llm_graph.llm.llm_call import LLMCall
from llm_graph.llm.response_functions import OpenAI_response_fn

# 1) Create an OpenAI client + response function
client = OpenAI()
response_fn = OpenAI_response_fn(client=client)

# 2) Define your custom prompt template
prompt_template = """
You are to find an answer to the query below and give your response in JSON format
with key 'answer'

{user_query}
"""

# 3) Create the LLMCall callable (to be used as a node function)
llm_call = LLMCall(
    response_fn=response_fn,
    prompt_template=prompt_template,
    query_key="user_query",
)

# Example usage (calling it with a state dict)
state = {"user_query": "What is the capital of France?"}
result_dict = llm_call(state)
print(result_dict)
```

In [9]:
# Another general question
r3 = session.execute({"user_query": "What does a ConditionalNode do and when would I use one?"})
Markdown(r3["state_dict"]["answer"])

A ConditionalNode is a workflow node that decides what to do next based on a condition function.

Conceptually, it evaluates its `condition_fn` against the current `state` when `execute(state)` is called, and uses that result to determine which downstream node the workflow should transition to.

You would use a ConditionalNode whenever your graph needs branching logic—for example:
- Route to different processing paths depending on what’s in the state (e.g., intent detected, tool needed, missing inputs)
- Stop early or take an error-handling branch if a prerequisite isn’t met
- Choose between multiple next steps based on intermediate results

In [10]:
# Another coding question
r4 = session.execute({"user_query": "Can you show me how to use the @tool_call decorator?"})
Markdown(r4["state_dict"]["answer"])

```python
from llm_graph.utils import tool_call
from llm_graph.core.nodes import FunctionalNode
from llm_graph.core.graphrunner import GraphRunner

# Wrap a normal function so it can be called with a state_dict.
# - input_key: state_dict key containing a dict of args for the function
# - output_key: key to store the function's return value under
@tool_call(input_key="tool_params", output_key="tool_output")
def query_strip(query: str):
    return query.strip()

# Prepare the tool parameters in the state under `tool_params`
def tool_prep(state):
    query = state["user_query"]
    return {"tool_params": {"query": query}}

prep_node = FunctionalNode(name="prep", func=tool_prep, next_node_name="tool")
tool_node = FunctionalNode(name="tool", func=query_strip)

graph_runner = GraphRunner(nodes=[prep_node, tool_node], start_node="prep")

result = graph_runner.execute({"user_query": "    This needs to be stripped. "})
print(result["tool_output"])  # "This needs to be stripped."
```

## Inspect the trace

In [11]:
# Show the execution path for the last query
graphrunner.print_trace()

Step 1
  Node: branch_llm
  Node type: FunctionalNode
  Input: {'message_history': [{'role': 'user', 'content': "\nYou are making a decision on the type of query given.\nThis will either be a coding query (asking for code examples, implementation details, usage snippets)\nor a general query (asking for conceptual explanations, descriptions, or comparisons).\n\nRespond ONLY with valid JSON with key 'query_type' set to either 'coding' or 'general'.\n\nThe query is: What is the difference between GraphRunner and SessionRunner?\n"}, {'role': 'assistant', 'content': '{"query_type":"general"}'}, {'role': 'user', 'content': '\nUse the following retrieved context to answer the question.\nContext :\n## Component : SessionRunner\n\nPurpose:\nExecuting multiple runs of a GraphRunner object, and maintaining state variables that are kept between runs. GraphRunner will not retain the `state_dict` between calls to `execute()`, so the SessionRunner is used to do this\n\nKey Responsibilities:\n- Run th

In [12]:
# Token usage across the whole session
print(f"In tokens : {session.in_tokens}")
print(f"Out tokens: {session.out_tokens}")

In tokens : 9058
Out tokens: 878
